# ChaoSSM — Phase 11: the ungated chaotic recurrence, trained end-to-end

## Why this notebook exists

Two independent reviewers identified the same gap. The paper's headline mechanism ("the learned
gate suppresses chaos") was tested only in a GATED architecture, where the `(1-g)h` skip path
gives gradient descent a cheap, built-in way to bypass the chaotic map. A reviewer put it sharply:
this makes the finding close to tautological — of course a residual-gated network learns to lean
on its residual path under a retention loss; that is what residual connections are for. The
decisive missing experiment is training the **ungated** chaotic recurrence end-to-end, where chaos
is the ONLY path (no skip, no gate), and asking whether training still suppresses it.

This matters for the paper's central claim in a specific way:

- If the ungated chaotic recurrence ALSO ends up with a negative conditional Lyapunov exponent
  after end-to-end training (no gate to blame), the paper's claim strengthens: training suppresses
  chaos through mechanisms beyond the gate (e.g. shrinking gamma, using W_mix to contract, or
  simply not exploiting the fold's expansive branch), and "trainability suppresses chaos" is a
  claim about training dynamics in general, not an artifact of one gate design.
- If the ungated arm CANNOT be trained to solve the task at all (diverges, or performs at floor
  regardless of learning rate) while remaining chaotic, that is also informative: it suggests
  chaos and trainability are near-incompatible for a different, more basic reason (optimization
  instability), which is itself worth reporting honestly rather than assumed.
- If the ungated arm trains successfully AND remains chaotic (conditional lambda > 0) while solving
  the task better than chance, that overturns the paper's central claim and must be reported as such.

## Design

The ungated cell is `h_t = fold(phi(A h_{t-1}) + B x_t)` -- identical to the gated cell of the main
paper MINUS the gate and skip path entirely (this is the same cell used in Phase 9/10's frozen fair
test, but here it TRAINS end-to-end rather than being frozen). We compare it against:
- the GATED chaotic arm (reproducing the main paper's result, for a same-notebook comparison)
- a periodic-rate ungated twin (the same causal ablation logic as Section 4.2, applied to the
  ungated cell)
- GRU and linear references

on copy and forecast, with the validated conditional-Lyapunov measurement (Benettin/QR, driven by
real inputs) applied to every trained arm -- not just accuracy.

## Pre-registered decision rule

Measure the conditional Lyapunov exponent of the trained UNGATED chaotic arm.
- lambda_conditional < 0  ->  suppression is not an artifact of the gate; training suppresses
  chaos even with no skip path available. The paper's mechanism claim is upheld in a STRONGER
  form (the gate is A mechanism, not THE only possible one).
- lambda_conditional > 0 (and task performance is non-trivial) -> the gate was doing the
  suppressing; an ungated architecture keeps its chaos. The paper's causal claim must be
  narrowed specifically to gated architectures.
- ungated arm fails to train / diverges regardless of chaos -> report as a separate, honest
  finding about optimization instability of chaotic recurrences without a skip path.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))
def setup_results_dir(project="ChaoSSM",phase="phase11_ungated"):
    try:
        from google.colab import drive; drive.mount("/content/drive",force_remount=False)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"./{project}_results/{phase}"); print(f"[info] not colab ({type(e).__name__})")
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase11_ungated


## Configuration

In [2]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64; n_layers:int=2; mlp_mult:int=2
    steps:int=4000; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0
    seeds:int=5
    eval_n:int=1500; eval_seed:int=12345
    copy_len:int=12; vocab:int=16
    fc_length:int=64; fc_bins:int=32
    r_cap_chaotic:float=4.0; r_cap_periodic:float=3.4   # same as main paper's ablation
    lr:dict=field(default_factory=lambda: dict(
        gru=3e-3, linear=1e-4,
        ungated_chaotic=1e-4, ungated_periodic=1e-4,
        gated_chaotic=1e-4))
    tasks:tuple=("copy","forecast")
    benettin_warmup:int=100; benettin_steps:int=400; benettin_traj:int=16
    arms:tuple=("gru","linear","ungated_chaotic","ungated_periodic","gated_chaotic")

CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.eval_n=40
    CFG.benettin_warmup=10; CFG.benettin_steps=30; CFG.benettin_traj=4
    print("*** SMOKE TEST ***")
print(json.dumps({k:v for k,v in asdict(CFG).items()},indent=2,default=str))

{
  "state_dim": 64,
  "d_model": 64,
  "n_layers": 2,
  "mlp_mult": 2,
  "steps": 4000,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "gamma_init": 1.0,
  "seeds": 5,
  "eval_n": 1500,
  "eval_seed": 12345,
  "copy_len": 12,
  "vocab": 16,
  "fc_length": 64,
  "fc_bins": 32,
  "r_cap_chaotic": 4.0,
  "r_cap_periodic": 3.4,
  "lr": {
    "gru": 0.003,
    "linear": 0.0001,
    "ungated_chaotic": 0.0001,
    "ungated_periodic": 0.0001,
    "gated_chaotic": 0.0001
  },
  "tasks": [
    "copy",
    "forecast"
  ],
  "benettin_warmup": 100,
  "benettin_steps": 400,
  "benettin_traj": 16,
  "arms": [
    "gru",
    "linear",
    "ungated_chaotic",
    "ungated_periodic",
    "gated_chaotic"
  ]
}


## 1. Cells

`UngatedChaosCell`: `h_t = fold(phi(A h_{t-1}) + B x_t)` -- no gate, no skip, chaos is the only
recurrent path. Its rate cap sets chaotic (r=4.0) vs periodic (r=3.4) exactly as the main paper's
ablation. `GatedChaosCell` reproduces the main paper's cell for a same-run comparison.

In [3]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaoticMap(nn.Module):
    def __init__(self,N,r_cap,gi=1.0):
        super().__init__(); self.r_cap=float(r_cap)
        self.w_r=nn.Parameter(torch.zeros(N))
        W=torch.empty(N,N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(gi)))))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)

class UngatedChaosCell(nn.Module):
    # No gate, no skip path -- chaos is the ONLY recurrent path. Trains end-to-end.
    def __init__(self,cfg,periodic=False):
        super().__init__(); self.N=cfg.state_dim
        r_cap = cfg.r_cap_periodic if periodic else cfg.r_cap_chaotic
        self.phi=ChaoticMap(cfg.state_dim,r_cap,cfg.gamma_init)
        self.A=nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9,1.1)))
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
    def step(self,h,x): return triangle_fold(self.phi(h@self.A.T)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)

class GatedChaosCell(nn.Module):
    # Reproduces the main paper cell (gate + (1-g)h skip) for direct in-notebook comparison.
    def __init__(self,cfg):
        super().__init__(); self.N=cfg.state_dim
        self.phi=ChaoticMap(cfg.state_dim,cfg.r_cap_chaotic,cfg.gamma_init)
        self.A=nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9,1.1)))
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.W_g=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
        self.b_g=nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self,h,x):
        g=torch.sigmoid(x@self.W_g.T+self.b_g)
        raw=g*self.phi(h@self.A.T)+(1-g)*h+x@self.B.T
        return triangle_fold(raw)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(cfg.state_dim,cfg.state_dim)/math.sqrt(cfg.state_dim))
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)

class GRUCellArm(nn.Module):
    def __init__(self,cfg,**kw): super().__init__(); self.N=cfg.state_dim; self.cell=nn.GRUCell(cfg.state_dim,cfg.state_dim)
    def step(self,h,x): return self.cell(x,h)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)

def make_cell(cfg,arm):
    if arm=="gru": return GRUCellArm(cfg)
    if arm=="linear": return LinearCell(cfg)
    if arm=="ungated_chaotic": return UngatedChaosCell(cfg,periodic=False)
    if arm=="ungated_periodic": return UngatedChaosCell(cfg,periodic=True)
    if arm=="gated_chaotic": return GatedChaosCell(cfg)
    raise ValueError(arm)

class Block(nn.Module):
    def __init__(self,cfg,arm):
        super().__init__(); self.arm=arm
        self.ln1=nn.LayerNorm(cfg.d_model); self.cell=make_cell(cfg,arm)
        self.ln2=nn.LayerNorm(cfg.d_model); h=cfg.d_model*cfg.mlp_mult
        self.mlp=nn.Sequential(nn.Linear(cfg.d_model,h),nn.GELU(),nn.Linear(h,cfg.d_model))
    def step(self,hs,x):
        u=self.ln1(x); hs=self.cell.step(hs,u); y=x+hs; y=y+self.mlp(self.ln2(y)); return hs,y
    def init_state(self,b,dev,dt): return self.cell.init_state(b,dev,dt)

class SeqLM(nn.Module):
    def __init__(self,cfg,arm,in_kind,in_dim,out_dim):
        super().__init__(); self.cfg=cfg; self.arm=arm; self.in_kind=in_kind
        if in_kind=="tokens": self.emb=nn.Embedding(in_dim,cfg.d_model)
        else: self.inp=nn.Linear(1,cfg.d_model)
        self.blocks=nn.ModuleList([Block(cfg,arm) for _ in range(cfg.n_layers)])
        self.ln_f=nn.LayerNorm(cfg.d_model); self.readout=nn.Linear(cfg.d_model,out_dim)
    def encode(self,x): return self.emb(x) if self.in_kind=="tokens" else self.inp(x.unsqueeze(-1))
    def forward(self,x,return_traj=False):
        e=self.encode(x); B,T=e.shape[0],e.shape[1]; dt=e.dtype
        st=[blk.init_state(B,e.device,dt) for blk in self.blocks]; outs=[]
        trajs=[[] for _ in self.blocks] if return_traj else None
        xseqs=[[] for _ in self.blocks] if return_traj else None
        for t in range(T):
            xt=e[:,t]
            for li,blk in enumerate(self.blocks):
                hi=xt; st[li],xt=blk.step(st[li],xt)
                if return_traj: trajs[li].append(st[li]); xseqs[li].append(blk.ln1(hi))
            outs.append(self.readout(self.ln_f(xt)))
        lg=torch.stack(outs,1)
        if not return_traj: return lg
        return lg,{li:(torch.stack(trajs[li],1),torch.stack(xseqs[li],1)) for li in range(len(self.blocks))}

## 2. Tasks (copy + forecast, matching the main paper)

In [4]:
V=CFG.vocab; SEP,PAD=V,V+1; VOCAB_SIZE=V+2
def make_copy(cfg,batch,device,gen):
    L=cfg.copy_len; src=torch.randint(0,V,(batch,L),device=device,generator=gen)
    inp=torch.cat([src,torch.full((batch,1),SEP,device=device),torch.full((batch,L),PAD,device=device)],1)
    tgt=torch.cat([torch.full((batch,L+1),-100,device=device),src],1)
    return inp,tgt,"tokens",VOCAB_SIZE,VOCAB_SIZE
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt,"scalar",1,cfg.fc_bins
TASK_FN=dict(copy=make_copy,forecast=make_forecast)
def build_eval(cfg,task):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return TASK_FN[task](cfg,cfg.eval_n,DEVICE,g)
# majority-class chance for forecast (invariant density is non-uniform; see main paper Sec 3.4)
@torch.no_grad()
def majority_class_chance(cfg,n=20000,seed=777):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(seed)
    _,tgt,_,_,_=make_forecast(cfg,n,DEVICE,g)
    counts=torch.bincount(tgt.reshape(-1),minlength=cfg.fc_bins)
    return float(counts.max())/float(counts.sum())
CHANCE=dict(copy=1.0/V)
CHANCE["forecast"]=majority_class_chance(CFG)
print(f"chance: copy={CHANCE['copy']:.4f}  forecast(majority-class)={CHANCE['forecast']:.4f}")
@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt,_,_,_=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]; m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

chance: copy=0.0625  forecast(majority-class)=0.1028


## 3. Trainer + validated conditional-Lyapunov measurement on the TRAINED model

In [5]:
from torch.func import jacrev
def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def train_arm(arm,task,seed,cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    inp,tgt,in_kind,in_dim,out_dim=TASK_FN[task](cfg,4,DEVICE,gen)
    model=SeqLM(cfg,arm,in_kind,in_dim,out_dim).to(DEVICE)
    lr=cfg.lr[arm]; opt=torch.optim.AdamW(model.parameters(),lr=lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    div=False
    for step in range(cfg.steps):
        inp,tgt,_,_,_=TASK_FN[task](cfg,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    return model,div

@torch.no_grad()
def real_input_sequence(model,cfg,seed=0,layer=0,length=None,task="copy"):
    length=length or (cfg.benettin_warmup+cfg.benettin_steps)
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(500+seed)
    inp,_,_,_,_=TASK_FN[task](cfg,1,DEVICE,g)
    e=model.encode(inp); dt=e.dtype
    st=[blk.init_state(1,DEVICE,dt) for blk in model.blocks]; cap=[]; T=inp.shape[1]
    for t in range(T):
        xt=e[:,t]
        for li,blk in enumerate(model.blocks):
            u=blk.ln1(xt)
            if li==layer: cap.append(u.squeeze(0).detach())
            h=blk.cell.step(st[li],u); st[li]=h; y=xt+h; y=y+blk.mlp(blk.ln2(y)); xt=y
    seq=torch.stack(cap,0)
    if seq.shape[0]<length: seq=seq.repeat((length//seq.shape[0])+1,1)
    return seq[:length]

@torch.no_grad()
def conditional_lambda(model,cfg,seed=0,layer=0,task="copy"):
    cell=model.blocks[layer].cell; N=cfg.state_dim
    total=cfg.benettin_warmup+cfg.benettin_steps
    seq=real_input_sequence(model,cfg,seed=seed,layer=layer,length=total,task=task)
    lam_maxes=[]
    for tr in range(cfg.benettin_traj):
        g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(9000+seed*97+tr)
        seqx=real_input_sequence(model,cfg,seed=9000+seed*97+tr,layer=layer,length=total,task=task)
        h=torch.full((N,),0.5,device=DEVICE)
        Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=DEVICE))[0]; logsum=torch.zeros(N,device=DEVICE)
        for t in range(total):
            x=seqx[t%seqx.shape[0]]
            J=jacrev(lambda z: cell.step(z.unsqueeze(0),x.unsqueeze(0)).squeeze(0))(h)
            h=cell.step(h.unsqueeze(0),x.unsqueeze(0)).squeeze(0)
            Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
            if t>=cfg.benettin_warmup: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
        lam_maxes.append(float(torch.sort(logsum/cfg.benettin_steps,descending=True)[0][0]))
    return float(np.mean(lam_maxes)), float(np.std(lam_maxes))

# estimator validation (logistic map, ln2) -- stand-alone check
def logistic_lyap(r=4.0,n=4000,warm=200,seed=0):
    rng=np.random.default_rng(seed); x=rng.uniform(0.1,0.9); s=0.0; c=0
    for t in range(n+warm):
        d=abs(r*(1-2*x)); x=r*x*(1-x)
        if t>=warm: s+=math.log(d+1e-12); c+=1
    return s/max(c,1)
KNOWN=logistic_lyap(); print(f"[validation] logistic r=4 lambda={KNOWN:.4f} vs ln2={math.log(2):.4f} err={abs(KNOWN-math.log(2)):.4f}")

[validation] logistic r=4 lambda=0.6929 vs ln2=0.6931 err=0.0002


## 4. Run: train every arm, measure accuracy AND conditional lambda for the chaotic arms

In [6]:
rows=[]; t0=time.time(); tot=len(CFG.arms)*len(CFG.tasks)*CFG.seeds; done=0
for arm in CFG.arms:
    for task in CFG.tasks:
        for sd in range(CFG.seeds):
            model,div=train_arm(arm,task,sd,CFG); done+=1
            evalset=build_eval(CFG,task); acc=eval_acc(model,evalset) if not div else float("nan")
            lam=lam_sd=float("nan")
            if not div and arm in ("ungated_chaotic","ungated_periodic","gated_chaotic"):
                lam,lam_sd=conditional_lambda(model,CFG,seed=sd,task=task)
            rows.append(dict(arm=arm,task=task,seed=sd,diverged=div,acc=acc,
                             conditional_lambda=lam,conditional_lambda_sd=lam_sd))
            fl=" DIV" if div else ""
            lamstr=f" lam={lam:+.3f}" if math.isfinite(lam) else ""
            print(f"[{done}/{tot}] {arm:18s} {task:9s} s={sd} acc={acc:.4f}{lamstr}{fl}")
RES=pd.DataFrame(rows); savetable(RES,"phase11_ungated_summary.csv")
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")
print(RES.groupby(["arm","task"])[["acc","conditional_lambda"]].mean().round(4).to_string())

[1/50] gru                copy      s=0 acc=0.9973
[2/50] gru                copy      s=1 acc=0.9978
[3/50] gru                copy      s=2 acc=0.9971
[4/50] gru                copy      s=3 acc=0.9959
[5/50] gru                copy      s=4 acc=0.9982
[6/50] gru                forecast  s=0 acc=0.9813
[7/50] gru                forecast  s=1 acc=0.9780
[8/50] gru                forecast  s=2 acc=0.9782
[9/50] gru                forecast  s=3 acc=0.9830
[10/50] gru                forecast  s=4 acc=0.9831
[11/50] linear             copy      s=0 acc=0.1094
[12/50] linear             copy      s=1 acc=0.0599
[13/50] linear             copy      s=2 acc=0.0674
[14/50] linear             copy      s=3 acc=0.0616
[15/50] linear             copy      s=4 acc=0.0872
[16/50] linear             forecast  s=0 acc=0.9236
[17/50] linear             forecast  s=1 acc=0.9381
[18/50] linear             forecast  s=2 acc=0.8935
[19/50] linear             forecast  s=3 acc=0.9188
[20/50] linear       

## 5. Analysis + figure

In [7]:
piv=RES.groupby(["arm","task"]).agg(acc_mean=("acc","mean"),acc_sd=("acc","std"),
                                     lam_mean=("conditional_lambda","mean"),
                                     lam_sd=("conditional_lambda","std")).round(4).reset_index()
print(piv.to_string(index=False)); savetable(piv,"phase11_summary_table.csv")

fig,axes=plt.subplots(1,len(CFG.tasks),figsize=(6.2*len(CFG.tasks),4.6))
if len(CFG.tasks)==1: axes=[axes]
for ax,task in zip(axes,CFG.tasks):
    sub=piv[piv.task==task].sort_values("acc_mean",ascending=False)
    ax.bar(range(len(sub)),sub.acc_mean,yerr=sub.acc_sd,capsize=3,
           color=["tab:purple" if a=="gru" else "gray" if a=="linear" else
                  "tab:green" if "ungated" in a else "tab:olive" for a in sub.arm])
    ax.axhline(CHANCE[task],color="r",ls="--",lw=1,label="chance")
    ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub.arm,rotation=30,ha="right",fontsize=8)
    ax.set_title(task); ax.set_ylabel("held-out accuracy"); ax.grid(alpha=.3,axis="y"); ax.legend(fontsize=8)
fig.suptitle("Phase 11: ungated chaotic recurrence, trained end-to-end")
fig.tight_layout(); savefig(fig,"phase11_bars.png"); plt.close(fig)

             arm     task  acc_mean  acc_sd  lam_mean  lam_sd
   gated_chaotic     copy    0.1637  0.0151   -0.0092  0.0704
   gated_chaotic forecast    0.5091  0.4256   -0.0041  0.5066
             gru     copy    0.9973  0.0009       NaN     NaN
             gru forecast    0.9807  0.0025       NaN     NaN
          linear     copy    0.0771  0.0211       NaN     NaN
          linear forecast    0.9175  0.0162       NaN     NaN
 ungated_chaotic     copy    0.0660  0.0022    0.3646  0.0580
 ungated_chaotic forecast    0.0537  0.0645    0.4014  0.1511
ungated_periodic     copy    0.0715  0.0097    0.3222  0.0355
ungated_periodic forecast    0.1352  0.0754    0.4279  0.0608
saved /content/drive/My Drive/ChaoSSM/results/phase11_ungated/phase11_summary_table.csv
saved /content/drive/My Drive/ChaoSSM/results/phase11_ungated/phase11_bars.png


## 6. Verdict

In [8]:
def get(arm,task,col):
    r=piv[(piv.arm==arm)&(piv.task==task)]
    return float(r[col].iloc[0]) if len(r) else float("nan")

ungated_lam_copy = get("ungated_chaotic","copy","lam_mean")
ungated_lam_fc   = get("ungated_chaotic","forecast","lam_mean")
ungated_acc_copy = get("ungated_chaotic","copy","acc_mean")
ungated_div = bool(RES[(RES.arm=="ungated_chaotic")].diverged.any())

est_ok = abs(KNOWN-math.log(2))<0.05
if ungated_div and RES[(RES.arm=="ungated_chaotic")].diverged.mean()>0.5:
    decision=("UNGATED ARM MOSTLY DIVERGES. Training the chaotic recurrence with no gate and no skip "
              "path is itself unstable for most seeds. This is a separate, honest finding: chaos and "
              "trainability may be incompatible here for an optimization-stability reason, not only a "
              "gate-suppression reason. Report both mechanisms; do not claim the gate-suppression "
              "story covers this case.")
elif math.isfinite(ungated_lam_copy) and ungated_lam_copy < 0 and math.isfinite(ungated_lam_fc) and ungated_lam_fc < 0:
    decision=("SUPPRESSION CONFIRMED WITHOUT A GATE. The ungated chaotic recurrence -- chaos is the "
              "ONLY recurrent path, no skip connection available -- still ends up with a NEGATIVE "
              "conditional Lyapunov exponent after end-to-end training on both tasks "
              f"(copy: {ungated_lam_copy:+.3f}, forecast: {ungated_lam_fc:+.3f}). This is a materially "
              "STRONGER result than the gated-only finding: it rules out the objection that gate "
              "suppression is merely a designed-in property of one architecture. Training suppresses "
              "chaos through some other means (e.g. shrinking gamma or using W_mix to contract) even "
              "when no skip path exists. Update the main paper: report this as a second, independent "
              "confirmation of the mechanism claim, and note the gate dose-response as ONE mechanism, "
              "not the only one.")
elif math.isfinite(ungated_lam_copy) and (ungated_lam_copy >= 0 or ungated_lam_fc >= 0):
    decision=("OBJECTION UPHELD: THE GATE WAS DOING THE SUPPRESSING. The ungated chaotic recurrence "
              f"retains a non-negative conditional exponent (copy: {ungated_lam_copy:+.3f}, "
              f"forecast: {ungated_lam_fc:+.3f}) after end-to-end training. This means the main paper's "
              "gated architecture suppressed chaos via its skip path specifically, not via a general "
              "property of training. The causal claim must be narrowed to GATED, residual chaotic "
              "recurrences; 'trainability suppresses chaos' as a general statement is NOT supported by "
              "this result and the abstract/title must be rescoped accordingly.")
else:
    decision="Inconclusive at these settings; inspect the per-arm table."

verdict=dict(smoke_test=SMOKE_TEST, estimator_ok=est_ok,
             ungated_chaotic_copy=dict(acc=ungated_acc_copy,lambda_mean=ungated_lam_copy),
             ungated_chaotic_forecast=dict(lambda_mean=ungated_lam_fc),
             ungated_diverged_frac=float(RES[RES.arm=="ungated_chaotic"].diverged.mean()),
             decision=decision)
savejson(verdict,"phase11_verdict.json")
print(decision); print(); print(json.dumps(verdict,indent=2,default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase11_ungated/phase11_verdict.json
OBJECTION UPHELD: THE GATE WAS DOING THE SUPPRESSING. The ungated chaotic recurrence retains a non-negative conditional exponent (copy: +0.365, forecast: +0.401) after end-to-end training. This means the main paper's gated architecture suppressed chaos via its skip path specifically, not via a general property of training. The causal claim must be narrowed to GATED, residual chaotic recurrences; 'trainability suppresses chaos' as a general statement is NOT supported by this result and the abstract/title must be rescoped accordingly.

{
  "smoke_test": false,
  "estimator_ok": true,
  "ungated_chaotic_copy": {
    "acc": 0.066,
    "lambda_mean": 0.3646
  },
  "ungated_chaotic_forecast": {
    "lambda_mean": 0.4014
  },
  "ungated_diverged_frac": 0.2,
  "decision": "OBJECTION UPHELD: THE GATE WAS DOING THE SUPPRESSING. The ungated chaotic recurrence retains a non-negative conditional exponent (copy: +0.36